# Trip Reason Classification

**Selected Machine Learning Exercise**

Binary classification of travel purpose from booking, route, timing, and passenger-related features.

In [1]:
import pandas as pd
import numpy as np

## inspecting the dataset

In [ ]:
test = pd.read_csv("data/test_data.csv")

In [ ]:
df = pd.read_csv("data/train_data.csv")
df.head()

In [4]:
df.shape

(101017, 22)

In [5]:
df.duplicated().sum()

np.int64(2)

In [6]:
df[df.duplicated(keep=False)]

,Created,CancelTime,DepartureTime,BillID,TicketID,ReserveStatus,UserID,Male,Price,CouponDiscount,...,Domestic,VehicleType,VehicleClass,TripReason,Vehicle,Cancel,HashPassportNumber_p,HashEmail,BuyerMobile,NationalCode
4302,2022-07-28 12:05:44.417,NaN,2022-08-08 21:40:00,38454499,7452955.0,3,NaN,False,1225000.0,0.0,...,1,SCANIA VIP 2+1 / شارژر یو اس بی / برق 220 ولت...,True,Int,Bus,0,NaN,NaN,623769789061,155777031
43959,2022-08-04 15:20:33.203,NaN,2022-08-06 04:40:00,38559562,1067895.0,5,735707.0,True,5870000.0,80000.0,...,1,NaN,False,Int,Plane,0,NaN,f65a8fac658175a186bbdfb03786c02e702d5df5c400af...,731735664474,804373868
76089,2022-07-28 12:05:44.417,NaN,2022-08-08 21:40:00,38454499,7452955.0,3,NaN,False,1225000.0,0.0,...,1,SCANIA VIP 2+1 / شارژر یو اس بی / برق 220 ولت...,True,Int,Bus,0,NaN,NaN,623769789061,155777031
82091,2022-08-04 15:20:33.203,NaN,2022-08-06 04:40:00,38559562,1067895.0,5,735707.0,True,5870000.0,80000.0,...,1,NaN,False,Int,Plane,0,NaN,f65a8fac658175a186bbdfb03786c02e702d5df5c400af...,731735664474,804373868


In [7]:
df = df.drop_duplicates()

In [8]:
df.duplicated().sum()

np.int64(0)

# feature engineering

## creating features and dropping them

In [9]:
x = df.drop(columns="TripReason")
y = df["TripReason"]

In [10]:
from sklearn.model_selection import train_test_split

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [11]:
id_cols = [
    "UserID",
    "BillID",
    "HashEmail",
    "BuyerMobile",
    "NationalCode",
    "HashPassportNumber_p"
]

# فقط روی train یاد بگیر
freq_maps = {col: x_train[col].value_counts(dropna=True) for col in id_cols}

def add_id_features(df, freq_maps):
    df = df.copy()

    for col, freq in freq_maps.items():

        # چند بار این ID در train دیده شده؟
        df[f"{col}_count"] = df[col].map(freq).fillna(0).astype(int)

        # آیا تکراری است؟
        df[f"{col}_repeated"] = (df[f"{col}_count"] > 1).astype(int)

    return df


x_train = add_id_features(x_train, freq_maps)
x_val   = add_id_features(x_val, freq_maps)
test  = add_id_features(test, freq_maps)

In [12]:
drop_cols = [
    "TicketID",
    "UserID",
    "BillID",
    "HashEmail",
    "BuyerMobile",
    "NationalCode",
    "HashPassportNumber_p"
]

x_train.drop(columns=drop_cols, inplace=True)
x_val.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)

## handling `NAN` values

In [13]:
x_train.isna().sum()

Created                              0
CancelTime                       68581
DepartureTime                        0
ReserveStatus                        0
Male                                 0
Price                                0
CouponDiscount                       0
From                                 0
To                                   0
Domestic                             0
VehicleType                       6012
VehicleClass                     30772
Vehicle                              0
Cancel                               0
UserID_count                         0
UserID_repeated                      0
BillID_count                         0
BillID_repeated                      0
HashEmail_count                      0
HashEmail_repeated                   0
BuyerMobile_count                    0
BuyerMobile_repeated                 0
NationalCode_count                   0
NationalCode_repeated                0
HashPassportNumber_p_count           0
HashPassportNumber_p_repe

In [14]:
x_train.isna().mean().mul(100)

Created                           0.000000
CancelTime                       84.864872
DepartureTime                     0.000000
ReserveStatus                     0.000000
Male                              0.000000
Price                             0.000000
CouponDiscount                    0.000000
From                              0.000000
To                                0.000000
Domestic                          0.000000
VehicleType                       7.439489
VehicleClass                     38.078503
Vehicle                           0.000000
Cancel                            0.000000
UserID_count                      0.000000
UserID_repeated                   0.000000
BillID_count                      0.000000
BillID_repeated                   0.000000
HashEmail_count                   0.000000
HashEmail_repeated                0.000000
BuyerMobile_count                 0.000000
BuyerMobile_repeated              0.000000
NationalCode_count                0.000000
NationalCod

* dropping `CancelTime`, since we have `cancel` column already

In [15]:
for df in [x_train, x_val, test]:
    df.drop(columns=["CancelTime"], inplace=True)

* nan values in `VehicleType` will be a category for itself
* for later one-hot encoding, we'll merge rare values

In [16]:
x_train["VehicleType"].value_counts(dropna=False)

VehicleType
NaN                                                                                     6012
4 ستاره اتوبوسي صبا                                                                     4994
3 ستاره 6 تخته پارسي                                                                    2400
25 نفره (VIP)                                                                           1783
4 ستاره 4 تخته غزال                                                                     1590
                                                                                        ... 
درساVIP تخت شو.شارژراختصاصی  پذیرایی میوه هرپنج سفر یک سفر رایگان                          1
SCANIA MARAL 2+1 /  /  /  /                                                                1
BENZ O500 2+1  / سیستم تهویه مطبوع / تخت شو                                                1
مارال مانیتوردار اینترنت                                                                   1
MARAL VIP مانیتوردار*تشریفاتی*امیرکبیر-کاراندیش/پذیرایی/هر

In [17]:
x_train["VehicleType"] = x_train["VehicleType"].fillna("Unknown")
x_val["VehicleType"] = x_val["VehicleType"].fillna("Unknown")
test["VehicleType"] = test["VehicleType"].fillna("Unknown")

In [18]:
freq = x_train["VehicleType"].value_counts()

rare = freq[freq < 20].index

for df in [x_train, x_val, test]:
    df["VehicleType"] = df["VehicleType"].replace(rare, "Rare")

* because of the strong relation between `VehicleClass` and `VehicleType`, we create a mapping for handling the missing values of `VehicleClass`

In [19]:
x_train["VehicleClass"].value_counts(dropna=False)

VehicleClass
True     37178
NaN      30772
False    12862
Name: count, dtype: int64

In [20]:
pd.crosstab(
    x_train["VehicleType"],
    x_train["VehicleClass"],
    normalize="index"
).sort_values(by=True, ascending=False)

VehicleClass,False,True
VehicleType,,
۲۵نفرهVOLVO-B9VIPتخت شو,0.0,1.0
,0.0,1.0
(MAN (V.I.P,0.0,1.0
(MAN(VIP مشترک با کاوه,0.0,1.0
(V.I.P)مارال مانیتوردار,0.0,1.0
...,...,...
اسکانیا /کلاسیک,1.0,0.0
VOLVO CLASSICUS 2+2 / سیستم تهویه مطبوع,1.0,0.0
آورو RJ85,1.0,0.0


In [21]:
pd.crosstab(x_train["VehicleType"], x_train["VehicleClass"])

VehicleClass,False,True
VehicleType,,
,0,30
(MAN (V.I.P,0,387
(MAN(VIP مشترک با کاوه,0,136
(V.I.P)مارال مانیتوردار,0,37
(VIP) 25,0,20
...,...,...
۲۵VIPنفره تخت شو (پذیرائی ),0,63
۲۵VIPنفره تخت شو (پذیرائی ویژه وماسک),0,40
۲۵نفره تخت شو(ازمسیر آزادراه جدید),0,27


In [22]:
vehicle_class_map = (
    x_train
    .dropna(subset=["VehicleClass", "VehicleType"])
    .groupby("VehicleType")["VehicleClass"]
    .agg(lambda x: x.mode().iloc[0])
)

# fallback فقط از train
vehicle_class_mode = x_train["VehicleClass"].mode().iloc[0]


for df in [x_train, x_val, test]:

    # اول با VehicleType
    df["VehicleClass"] = df["VehicleClass"].fillna(
        df["VehicleType"].map(vehicle_class_map)
    )

    # هرچی باقی موند
    df["VehicleClass"] = df["VehicleClass"].fillna(
        vehicle_class_mode
    )

In [23]:
x_train.isna().sum()

Created                          0
DepartureTime                    0
ReserveStatus                    0
Male                             0
Price                            0
CouponDiscount                   0
From                             0
To                               0
Domestic                         0
VehicleType                      0
VehicleClass                     0
Vehicle                          0
Cancel                           0
UserID_count                     0
UserID_repeated                  0
BillID_count                     0
BillID_repeated                  0
HashEmail_count                  0
HashEmail_repeated               0
BuyerMobile_count                0
BuyerMobile_repeated             0
NationalCode_count               0
NationalCode_repeated            0
HashPassportNumber_p_count       0
HashPassportNumber_p_repeated    0
dtype: int64

In [24]:
test.isna().sum()

Created                          0
DepartureTime                    0
ReserveStatus                    0
Male                             0
Price                            0
CouponDiscount                   0
From                             0
To                               0
Domestic                         0
VehicleType                      0
VehicleClass                     0
Vehicle                          0
Cancel                           0
UserID_count                     0
UserID_repeated                  0
BillID_count                     0
BillID_repeated                  0
HashEmail_count                  0
HashEmail_repeated               0
BuyerMobile_count                0
BuyerMobile_repeated             0
NationalCode_count               0
NationalCode_repeated            0
HashPassportNumber_p_count       0
HashPassportNumber_p_repeated    0
dtype: int64

In [25]:
x_val.isna().sum()

Created                          0
DepartureTime                    0
ReserveStatus                    0
Male                             0
Price                            0
CouponDiscount                   0
From                             0
To                               0
Domestic                         0
VehicleType                      0
VehicleClass                     0
Vehicle                          0
Cancel                           0
UserID_count                     0
UserID_repeated                  0
BillID_count                     0
BillID_repeated                  0
HashEmail_count                  0
HashEmail_repeated               0
BuyerMobile_count                0
BuyerMobile_repeated             0
NationalCode_count               0
NationalCode_repeated            0
HashPassportNumber_p_count       0
HashPassportNumber_p_repeated    0
dtype: int64

## creating a vacation boolean feature

In [26]:
import holidays

In [27]:
years = set()

for df in [x_train, x_val, test]:
    dates = pd.to_datetime(df["DepartureTime"])
    years.update(dates.dt.year.unique())

# تعطیلات رسمی ایران
iran_holidays = holidays.country_holidays(
    "IR",
    years=sorted(years)
)

holiday_dates = set(iran_holidays.keys())


def add_holiday_features(df):
    df = df.copy()

    departure = pd.to_datetime(df["DepartureTime"])
    date = departure.dt.date

    # تعطیلات رسمی
    df["IsOfficialHoliday"] = (
        date.isin(holiday_dates)
    ).astype(int)

    # جمعه، تعطیلی هفتگی رسمی ایران
    df["IsFriday"] = (
        departure.dt.dayofweek == 4
    ).astype(int)

    # تعطیل رسمی یا جمعه
    df["IsHoliday"] = (
        (df["IsOfficialHoliday"] == 1) |
        (df["IsFriday"] == 1)
    ).astype(int)

    return df


x_train = add_holiday_features(x_train)
x_val   = add_holiday_features(x_val)
test  = add_holiday_features(test)

In [28]:
def add_weekend_features(df):
    df = df.copy()

    departure = pd.to_datetime(df["DepartureTime"])

    df["IsThursday"] = (departure.dt.dayofweek == 3).astype(int)
    df["IsFriday"] = (departure.dt.dayofweek == 4).astype(int)

    df["IsWeekend"] = (
        df["IsThursday"] | df["IsFriday"]
    ).astype(int)

    return df


x_train = add_weekend_features(x_train)
x_val   = add_weekend_features(x_val)
test  = add_weekend_features(test)

In [29]:
x_train.head()

,Created,DepartureTime,ReserveStatus,Male,Price,CouponDiscount,From,To,Domestic,VehicleType,...,BuyerMobile_repeated,NationalCode_count,NationalCode_repeated,HashPassportNumber_p_count,HashPassportNumber_p_repeated,IsOfficialHoliday,IsFriday,IsHoliday,IsThursday,IsWeekend
99941,2022-03-20 18:03:33.467,2022-03-21 00:30:00,3,True,700000.0,0.0,صومعه سرا,تهران,1,Rare,...,1,2,1,0,0,1,0,1,0,0
78313,2022-08-30 10:14:17.300,2022-08-30 16:00:00,3,True,900000.0,0.0,تهران,اراک,1,25 نفره (VIP),...,0,1,0,0,0,0,0,0,0,0
71761,2022-07-13 00:48:47.137,2022-07-14 05:00:00,3,True,520000.0,0.0,سمنان,تهران,1,classicus 2+2,...,1,3,1,0,0,0,0,0,1,1
60206,2022-05-06 21:10:50.080,2022-05-08 12:30:00,3,True,1350000.0,0.0,همدان,اصفهان,1,SCANIA DORSA VIP 2+1 / مانیتوردار / پذیرایی /...,...,1,2,1,0,0,0,0,0,0,0
32195,2022-09-01 04:06:52.913,2022-09-01 08:00:00,2,False,1553000.0,0.0,تهران,مشهد,1,4 ستاره اتوبوسي صبا,...,1,1,0,0,0,0,0,0,1,1


In [30]:
def add_datetime_features(df):
    df = df.copy()

    df["Created"] = pd.to_datetime(df["Created"])
    df["DepartureTime"] = pd.to_datetime(df["DepartureTime"])

    # زمان حرکت
    df["DepartureHour"] = df["DepartureTime"].dt.hour
    df["DepartureDayOfWeek"] = df["DepartureTime"].dt.dayofweek
    df["DepartureMonth"] = df["DepartureTime"].dt.month

    # زمان رزرو
    df["CreatedHour"] = df["Created"].dt.hour
    df["CreatedDayOfWeek"] = df["Created"].dt.dayofweek
    df["CreatedMonth"] = df["Created"].dt.month

    # فاصله رزرو تا حرکت
    time_diff = df["DepartureTime"] - df["Created"]

    df["HoursUntilDeparture"] = (
        time_diff.dt.total_seconds() / 3600
    )

    df["DaysUntilDeparture"] = (
        time_diff.dt.total_seconds() / (3600 * 24)
    )

    return df


x_train = add_datetime_features(x_train)
x_val   = add_datetime_features(x_val)
test  = add_datetime_features(test)

In [31]:
datetime_cols = [
    "Created",
    "DepartureTime"
]

for df in [x_train, x_val, test]:
    df.drop(columns=datetime_cols, inplace=True)

In [32]:
x_train.head()

,ReserveStatus,Male,Price,CouponDiscount,From,To,Domestic,VehicleType,VehicleClass,Vehicle,...,IsThursday,IsWeekend,DepartureHour,DepartureDayOfWeek,DepartureMonth,CreatedHour,CreatedDayOfWeek,CreatedMonth,HoursUntilDeparture,DaysUntilDeparture
99941,3,True,700000.0,0.0,صومعه سرا,تهران,1,Rare,False,Bus,...,0,0,0,0,3,18,6,3,6.440704,0.268363
78313,3,True,900000.0,0.0,تهران,اراک,1,25 نفره (VIP),True,Bus,...,0,0,16,1,8,10,1,8,5.761861,0.240078
71761,3,True,520000.0,0.0,سمنان,تهران,1,classicus 2+2,True,Bus,...,1,1,5,3,7,0,2,7,28.186906,1.174454
60206,3,True,1350000.0,0.0,همدان,اصفهان,1,SCANIA DORSA VIP 2+1 / مانیتوردار / پذیرایی /...,True,Bus,...,0,0,12,6,5,21,4,5,39.319422,1.638309
32195,2,False,1553000.0,0.0,تهران,مشهد,1,4 ستاره اتوبوسي صبا,True,Train,...,1,1,8,3,9,4,3,9,3.885302,0.161888


* cities features

In [33]:
# فقط از train یاد می‌گیریم
route_freq = (
    x_train
    .groupby(["From", "To"])
    .size()
)

from_freq = x_train["From"].value_counts()
to_freq   = x_train["To"].value_counts()


def add_location_features(df):
    df = df.copy()

    # محبوبیت مبدا
    df["OriginFrequency"] = (
        df["From"].map(from_freq)
        .fillna(0)
    )

    # محبوبیت مقصد
    df["DestinationFrequency"] = (
        df["To"].map(to_freq)
        .fillna(0)
    )

    # محبوبیت مسیر
    routes = pd.MultiIndex.from_frame(
        df[["From", "To"]]
    )

    df["RouteFrequency"] = (
        route_freq.reindex(routes)
        .fillna(0)
        .to_numpy()
    )

    return df


x_train = add_location_features(x_train)
x_val   = add_location_features(x_val)
test  = add_location_features(test)

In [34]:
north_cities = [
    # Gilan
    "آستارا",
    "بندرانزلی",
    "رشت",
    "رودسر",
    "صومعه سرا",
    "طوالش",
    "فومن",
    "لاهیجان (گیلان )",
    "لنگرود",
    "ماسال",

    # Mazandaran
    "آمل",
    "بابل",
    "بابلسر",
    "بهشهر",
    "تنکابن",
    "رامسر",
    "ساری",
    "شیرگاه",
    "عباس آباد(مازندران )",
    "قائمشهر",
    "قایم شهر",
    "محمودآباد (مازندران )",
    "نور",
    "نوشهر",
    "نکا",
    "پل سفید",
    "چالوس",

    # Golestan
    "بندر ترکمن",
    "کلاله",
    "گرگان",
    "گنبدکاووس",
]

In [35]:
for df in [x_train, x_val, test]:
    df["IsNorthDestination"] = (
        df["To"].isin(north_cities)
    ).astype(int)

In [36]:
x_train.loc[
    x_train["IsNorthDestination"] == 1,
    "To"
].value_counts()

To
رشت                      1162
ساری                      625
گرگان                     606
بابل                      174
بابلسر                    165
رامسر                     135
تنکابن                    134
بندرانزلی                 112
گنبدکاووس                 103
نور                        84
چالوس                      74
لاهیجان (گیلان )           73
آمل                        69
لنگرود                     68
رودسر                      63
نوشهر                      53
آستارا                     44
فومن                       32
محمودآباد (مازندران )      27
طوالش                      22
قایم شهر                   17
بهشهر                      15
ماسال                      15
عباس آباد(مازندران )       13
قائمشهر                    13
شیرگاه                     11
صومعه سرا                   8
کلاله                       8
پل سفید                     5
نکا                         3
بندر ترکمن                  1
Name: count, dtype: int64

## handling categorical columns and grouping all of them

In [37]:
numeric = [
    "Price",
    "CouponDiscount",

    "UserID_count",
    "BillID_count",
    "HashEmail_count",
    "BuyerMobile_count",
    "NationalCode_count",
    "HashPassportNumber_p_count",

    "OriginFrequency",
    "DestinationFrequency",
    "RouteFrequency",

    "HoursUntilDeparture",

    "DepartureHour_sin",
    "DepartureHour_cos",
    "DepartureDayOfWeek_sin",
    "DepartureDayOfWeek_cos",
    "DepartureMonth_sin",
    "DepartureMonth_cos",

    "CreatedHour_sin",
    "CreatedHour_cos",
    "CreatedDayOfWeek_sin",
    "CreatedDayOfWeek_cos",
    "CreatedMonth_sin",
    "CreatedMonth_cos",
]

In [38]:
boolean = [
    "Male",
    "Domestic",
    "Cancel",
    "VehicleClass",

    # ID features
    "UserID_repeated",
    "BillID_repeated",
    "HashEmail_repeated",
    "BuyerMobile_repeated",
    "NationalCode_repeated",
    "HashPassportNumber_p_repeated",

    # Holiday/weekend
    "IsOfficialHoliday",
    "IsHoliday",
    "IsThursday",
    "IsFriday",
    "IsWeekend",

    # Location
    "IsNorthDestination",
]

In [39]:
cat_1hot = [
    "ReserveStatus",
    "Vehicle",
]

In [40]:
cyclic = [
    "DepartureHour",
    "DepartureDayOfWeek",
    "DepartureMonth",
    "CreatedHour",
    "CreatedDayOfWeek",
    "CreatedMonth",
]

In [41]:
cat_high_card = [
    "VehicleType",
]

In [42]:
import numpy as np

cyclic_periods = {
    "DepartureHour": 24,
    "DepartureDayOfWeek": 7,
    "DepartureMonth": 12,

    "CreatedHour": 24,
    "CreatedDayOfWeek": 7,
    "CreatedMonth": 12,
}

for df in [x_train, x_val, test]:
    for col, period in cyclic_periods.items():

        df[f"{col}_sin"] = np.sin(
            2 * np.pi * df[col] / period
        )

        df[f"{col}_cos"] = np.cos(
            2 * np.pi * df[col] / period
        )

In [43]:
cyclic_periods = {
    "DepartureHour": 24,
    "DepartureDayOfWeek": 7,
    "DepartureMonth": 12,
    "CreatedHour": 24,
    "CreatedDayOfWeek": 7,
    "CreatedMonth": 12,
}

for df in [x_train, x_val, test]:
    for col, period in cyclic_periods.items():
        df[f"{col}_sin"] = np.sin(2 * np.pi * df[col] / period)
        df[f"{col}_cos"] = np.cos(2 * np.pi * df[col] / period)

    df.drop(columns=list(cyclic_periods.keys()), inplace=True)

In [45]:
for col in cyclic_periods:
    numeric.extend([
        f"{col}_sin",
        f"{col}_cos"
    ])

In [46]:
assigned_cols = (
    numeric
    + boolean
    + cat_1hot
    + cat_high_card
    + cyclic
)

uncovered = [
    col for col in x_train.columns
    if col not in assigned_cols
]

print("Uncovered:", uncovered)

Uncovered: ['From', 'To', 'DaysUntilDeparture']


In [47]:
for df in [x_train, x_val, test]:
    df.drop(
        columns=[
            "From",
            "To",
            "DaysUntilDeparture"
        ],
        inplace=True
    )

In [48]:
assigned_cols = (
    numeric
    + boolean
    + cat_1hot
    + cat_high_card
    + cyclic
)

uncovered = [
    col for col in x_train.columns
    if col not in assigned_cols
]

print("Uncovered:", uncovered)

Uncovered: []


## creating pipelines

In [49]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

In [50]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesClassifier


numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

high_card_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=20
    )),
])


preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric),
    ("bool", "passthrough", boolean),
    ("cat", cat_pipeline, cat_1hot),
    ("high_card", high_card_pipeline, cat_high_card),
])


model_pipeline = Pipeline([
    ("preprocessor", preprocessor),

    ("model", ExtraTreesClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

In [51]:
y_train.value_counts()

TripReason
Work    45158
Int     35654
Name: count, dtype: int64

In [52]:
model_pipeline.fit(x_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('bool', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [53]:
from sklearn.metrics import classification_report, accuracy_score

y_pred = model_pipeline.predict(x_val)

print("Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))

Accuracy: 0.8716527248428452
              precision    recall  f1-score   support

         Int       0.93      0.77      0.84      8913
        Work       0.84      0.95      0.89     11290

    accuracy                           0.87     20203
   macro avg       0.88      0.86      0.87     20203
weighted avg       0.88      0.87      0.87     20203



In [54]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesClassifier

# کل داده آموزشی
x_full = pd.concat(
    [x_train, x_val],
    axis=0,
    ignore_index=True
)

y_full = pd.concat(
    [y_train, y_val],
    axis=0,
    ignore_index=True
)

print(x_full.shape)
print(y_full.shape)

(101015, 43)
(101015,)


In [55]:
final_model = Pipeline([
    ("preprocessor", preprocessor),

    ("model", ExtraTreesClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

final_model.fit(x_full, y_full)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('bool', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [56]:
test_pred = final_model.predict(test)

print(test_pred[:10])
print(pd.Series(test_pred).value_counts())

['Int' 'Work' 'Work' 'Work' 'Work' 'Work' 'Int' 'Work' 'Int' 'Work']
Work    27286
Int     16007
Name: count, dtype: int64


In [ ]:
from pathlib import Path
Path("outputs").mkdir(exist_ok=True)

submission = pd.DataFrame({
    "TripReason": test_pred
})

submission.to_csv("outputs/submission.csv", index=False)

submission.head()

In [ ]:
import joblib

joblib.dump(final_model, "outputs/trip_reason_pipeline.joblib")

In [ ]:
import zipfile

artifact_files = [
    "outputs/submission.csv",
    "outputs/trip_reason_pipeline.joblib",
]

with zipfile.ZipFile("outputs/model_artifacts.zip", mode="w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_name in artifact_files:
        zf.write(file_name, arcname=Path(file_name).name)

print("Created outputs/model_artifacts.zip")